In [8]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from mrmr import mrmr_classif
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

In [9]:
X_train = pd.read_excel("Data/X_train.xlsx").iloc[:, 1:]
y_train = pd.read_excel("Data/y_train.xlsx").iloc[:, 1].values
X_test  = pd.read_excel("Data/X_test.xlsx").iloc[:, 1:]
y_test  = pd.read_excel("Data/y_test.xlsx").iloc[:, 1].values

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [10]:
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=10):
        self.K = K
        self.selected_features_ = None

    def fit(self, X, y):
        # Convert through numpy first to guarantee a clean 0-based RangeIndex on
        # both X_df and y_s. Without this, CV-sliced DataFrames keep their original
        # non-contiguous index while pd.Series(y) gets 0..n, causing mrmr's internal
        # boolean indexer to raise IndexingError on index misalignment.
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        y_s  = pd.Series(np.array(y).ravel())
        self.selected_features_ = mrmr_classif(X_df, y_s, K=self.K)
        return self

    def transform(self, X):
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        return X_df[self.selected_features_].values

In [11]:
models = {
    "XGB": XGBClassifier(eval_metric="logloss", nthread=1),
    "LR": LogisticRegression(max_iter=2000),
    "RF": RandomForestClassifier(n_jobs=1),
    "MLP": MLPClassifier(early_stopping=True),
    "SVM": SVC(probability=True, cache_size=2000),
    "AB": AdaBoostClassifier(),
    "ET": ExtraTreesClassifier(n_jobs=1),
    # min_child_samples>=5 prevents -inf gain loops on small leaves.
    # force_col_wise avoids a Windows threading deadlock in LightGBM.
    "LGBM": LGBMClassifier(verbose=-1, n_jobs=1, min_child_samples=5,
                   min_split_gain=1e-4, force_col_wise=True)
}

param_grids = {
    "XGB": {'model__max_depth':[2,3], 'model__eta':[0.01,0.03,0.3], 'model__n_estimators':[30,50,100], 'model__reg_lambda':[1,3,8]},
    "LR": {"model__C":[1e-4,1e-3,1e-2,0.1,1,10]},
    "RF": {'model__max_depth':[2,3], 'model__min_samples_leaf':[2,3,4], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "MLP": {"model__hidden_layer_sizes":[(10,)], "model__alpha": [0.001,0.01,0.1,1], 'model__max_iter':[2000]},
    "SVM": {"model__C":[1e-3,0.01,0.1,1], 'model__kernel':['rbf','linear'], 'model__gamma':[0.01,0.1,1,10,100]},
    "AB": {'model__learning_rate':[0.001,0.01,0.1], 'model__estimator':[DecisionTreeClassifier(max_depth=i) for i in range(2,4)], 'model__n_estimators':[30,50,100]},
    "ET": {'model__max_depth':[2,3], 'model__min_samples_leaf':[3,4,5], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "LGBM": {'model__learning_rate':[0.001,0.01,0.1,1], 'model__max_depth':[2,3], 'model__num_leaves':[5,10,20,31], 'model__n_estimators':[30,50,100]}
}

In [12]:
def get_selectors(model):
    """Return a fresh set of feature selectors for the given model.

    clone(model) gives each selector its own independent estimator so fitted
    state never leaks between the selector and the pipeline's final model.
    n_jobs=1 on all selectors prevents nested parallelism with
    RandomizedSearchCV(n_jobs=-1) which owns all cores at the outer level.
    tol=1e-3 lets sequential selectors stop early when marginal gain is tiny.
    """
    selectors = {}

    # RFECV — tree-based / gradient-boosted models only
    if isinstance(model, (RandomForestClassifier, ExtraTreesClassifier,
                          XGBClassifier, LGBMClassifier)):
        selectors["rfecv"] = RFECV(
            estimator=clone(model), step=1, cv=3,
            scoring="f1_weighted", min_features_to_select=3, n_jobs=1,
        )

    # Sequential selectors — tol stops when marginal F1 gain < 0.1 %
    selectors["ffs"] = SequentialFeatureSelector(
        clone(model), direction="forward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )
    selectors["bfs"] = SequentialFeatureSelector(
        clone(model), direction="backward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )

    selectors["mrmr"] = MRMRSelector(K=10)
    selectors["none"] = "passthrough"

    return selectors

In [13]:
def evaluate(model, X, y):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X)
        if y_proba.shape[1] == 2:
            auc = roc_auc_score(y, y_proba[:,1])
        else:
            auc = roc_auc_score(y, y_proba, multi_class="ovr")
    else:
        auc = np.nan

    return {
        "f1_weighted": f1_score(y, y_pred, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "auc": auc,
        "sensitivity": recall_score(y, y_pred),
        "specificity": recall_score(y, y_pred, pos_label=0)
    }

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

os.makedirs("Results", exist_ok=True)
results = []

for model_name, model in models.items():
    param_grid = param_grids[model_name]
    print(f"\nModel: {model_name}")

    for fs_name, fs in get_selectors(model).items():
        model_path = f"Results/{model_name}_{fs_name}.joblib"

        if os.path.exists(model_path):
            print(f"  FS: {fs_name}  [skipped — already exists]")
            continue

        print(f"  FS: {fs_name}")

        pipe = Pipeline([("fs", fs), ("model", model)])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=20,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            random_state=42,
        )

        try:
            search.fit(X_train, y_train, model__sample_weight=sample_weights)
        except Exception:
            search.fit(X_train, y_train)
        
        # Best estimator
        best_model = search.best_estimator_

        n_feats = best_model.named_steps["model"].n_features_in_
        if fs_name != "none":
            print(f"    Number of selected features: {n_feats}")
        
        hyperp = str(search.best_params_)
        print(f"    Hyperparameters of the best estimator:\n    {hyperp}")
        
        cv_f1_mean = search.best_score_
        cv_f1_std = search.cv_results_['std_test_score'][search.best_index_]

        # Evaluation
        train_scores = evaluate(best_model, X_train, y_train)
        test_scores  = evaluate(best_model, X_test,  y_test)

        print("    f1 score:")
        print(f"     - cv: {cv_f1_mean} ({cv_f1_std})")
        print(f"     - train: {train_scores['f1_weighted']}")
        print(f"     - test: {test_scores['f1_weighted']}")

        results.append({
            "model": model_name,
            "fs": fs_name,
            "n features": n_feats,
            "hyperparameters": hyperp,
            "cv_f1_mean": cv_f1_mean,
            "cv_f1_std": cv_f1_std,
            **{f"train {k}": v for k, v in train_scores.items()},
            **{f"test {k}": v for k, v in test_scores.items()},
        })

        # Save search and model
        #joblib.dump(search, f"Results/search_{model_name}_{fs_name}.joblib")
        joblib.dump(best_model, model_path)

        print("\n")
        # Incremental save
        pd.DataFrame(results).to_excel("Results/classif_results.xlsx", index=False)

print("\nDone.")


Model: XGB
  FS: rfecv
    Number of selected features: 11
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 30, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.7017700595460467 (0.04307728270747392)
     - train: 0.8080702598852076
     - test: 0.5957182422908578


  FS: ffs
    Number of selected features: 5
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 1, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6129665620941974 (0.02281104427918486)
     - train: 0.6614195013330814
     - test: 0.6487673968670037


  FS: bfs
    Number of selected features: 22
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 1, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__eta': 0.3}
    f1 score:
     - cv: 0.6958674400204079 (0.054801794056756795)
     - train: 0.9756765640796934
     - test: 0.6235365091577044


  FS: mrmr


100%|██████████| 10/10 [00:04<00:00,  2.46it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 1, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6837688746187626 (0.035996150204133416)
     - train: 0.7698552711607541
     - test: 0.6422018348623854


  FS: none
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.698849772130066 (0.04314141226068616)
     - train: 0.7672957167694009
     - test: 0.6419005818817187



Model: LR
  FS: ffs
    Number of selected features: 4
    Hyperparameters of the best estimator:
    {'model__C': 0.1}
    f1 score:
     - cv: 0.6330993872699913 (0.047732182712053935)
     - train: 0.6994580599863967
     - test: 0.650551141051169


  FS: bfs
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__C': 10}
    f1 score:
     - cv: 0.6739930444

100%|██████████| 10/10 [00:02<00:00,  3.79it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__C': 0.01}
    f1 score:
     - cv: 0.678802421288917 (0.0323842014019425)
     - train: 0.6828787282572117
     - test: 0.6689431862590022


  FS: none
    Hyperparameters of the best estimator:
    {'model__C': 10}
    f1 score:
     - cv: 0.6717163440052434 (0.04388266840861653)
     - train: 0.7158282358422731
     - test: 0.6512587650202329



Model: RF
  FS: rfecv
    Number of selected features: 22
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 3, 'model__min_samples_leaf': 2, 'model__max_depth': 3}
    f1 score:
     - cv: 0.6962333363035298 (0.03845926528024928)
     - train: 0.7701175099294221
     - test: 0.6881259073919623


  FS: ffs
    Number of selected features: 5
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_depth': 3}
  

...

    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=2)}
    f1 score:
     - cv: 0.6905126879148813 (0.04579735245528341)
     - train: 0.7393689255758221
     - test: 0.6514348376733697


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=3)}
    f1 score:
     - cv: 0.6922508608329504 (0.0218303016104043)
     - train: 0.8292423921384986
     - test: 0.6229655489702579



Model: ET
  FS: rfecv
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 3, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6720619266292511 (0.03534840260764766)
     - train: 0.6739627453514956
     - test: 0.6389094942442664


  FS: ffs
    

100%|██████████| 10/10 [00:02<00:00,  3.89it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 4, 'model__min_samples_leaf': 5, 'model__max_depth': 3}
    f1 score:
     - cv: 0.6719859059232673 (0.06694069451302496)
     - train: 0.6851599603355785
     - test: 0.6188958635119909


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 3}
    f1 score:
     - cv: 0.6715972790300049 (0.0518143508855883)
     - train: 0.6956985623280627
     - test: 0.6769911745626803



Model: LGBM
  FS: rfecv
    Number of selected features: 20
    Hyperparameters of the best estimator:
    {'model__num_leaves': 5, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6977221130186007 (0.038505309482137914)
     - train: 0.8540114059882623
     - test: 0.6501999529522464


  FS: ffs
    Number of

100%|██████████| 10/10 [00:02<00:00,  3.93it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__num_leaves': 5, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6707242511053613 (0.03500818499481887)
     - train: 0.8107112841155394
     - test: 0.5961943594971116


  FS: none
    Hyperparameters of the best estimator:
    {'model__num_leaves': 5, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6925568055974185 (0.05064582396683122)
     - train: 0.8513220323054423
     - test: 0.6413574734107332



Done.
